# 02. Prophet 시계열 분석 — 변수군별 관계와 요일 흐름

**이 노트북이 답하는 질문 3가지**
| # | 질문 | 위치 |
| --- | --- | --- |
| Q1 | Prophet으로 이 과제에서 **무엇을 할 수 있고 무엇은 못 하나?** | §0, §4 |
| Q2 | 금융·포털·쇼핑·날씨 **변수군별로 Target과 어떤 관계인지 어떻게 확인했나?** | §2, §3 |
| Q3 | **요일별로 바뀌는 흐름**을 어떻게 고려했나? | §1, §5 |

**원칙** (01 EDA·train.ipynb와 동일)
- §1~§3 분석은 **Train만** 사용한다 (Test는 보지 않음).
- §4 예측 실험은 train.ipynb와 **같은 시계열 CV**(5-fold, gap=3)로 평가한다. Prophet은 매 fold에서 **그 fold의 Train 날짜까지만** 학습한다 → 미래 정보 사용 없음.
- 실행 커널: `Python (llm)` — `prophet`이 설치된 환경 (`%pip install prophet`).

## 0. Prophet이란? — 이 과제에 맞는 부분과 안 맞는 부분

Prophet(Meta)은 시계열을 **더하기로 분해**해서 예측하는 모델이다.

$$y(t) = \underbrace{g(t)}_{\text{추세}} + \underbrace{s_{week}(t)}_{\text{요일 패턴}} + \underbrace{s_{year}(t)}_{\text{연 계절성}} + \varepsilon_t$$

| 구성요소 | 의미 | 이 과제에서 |
| --- | --- | --- |
| 추세 `trend` | 몇 달에 걸친 큰 흐름 (상승·하락) | 금융·쇼핑의 장기 이동 |
| 요일 `weekly` | 일~토 7일 주기로 반복되는 패턴 | **T2(쇼핑 클릭 지수)의 핵심** |
| 연 계절성 `yearly` | 1년 주기 (여름·겨울 등) | 날씨, 계절 상품 |

**맞는 부분** ✅
- 요일·추세·계절성을 **자동으로 분리**해서 보여준다 → 변수군의 성격을 한눈에 비교 (§2)
- 과거 데이터만으로 **미래 값을 예측**할 수 있다 → 쇼핑 클릭의 3일 뒤 값을 예측해 T2 피처로 활용 (§4)

**안 맞는 부분** ⚠️
- Prophet은 **연속값**을 예측한다. 우리 Target은 0/1(상승 여부)이다.
- Target의 원변수(`FI_KOSPI`, `SH2_쇼핑2_식품_건강식품`)는 **데이터에서 빠져 있다** → 원변수를 직접 예측할 수 없고 **대리변수**(`SH1_쇼핑1_식품` 등)를 써야 한다.
- 주가(KOSPI)는 요일·계절 패턴이 거의 없는 **무작위 보행**에 가깝다 → Prophet이 잘하는 영역이 아니다.

**날짜 처리**: 데이터에는 실제 날짜가 없고 `Time_step`(순번)과 `Weekday`(1~7)만 있다. Prophet은 날짜가 필요하므로 **요일이 맞는 가상 날짜**를 만든다(§1에서 근거 확인). 연 계절성의 '몇 월'은 가상이지만 **1년 주기 자체는 그대로** 유효하다.

In [ ]:
# ── 설정 ─────────────────────────────────────────────
import warnings, logging, time
warnings.filterwarnings('ignore')
logging.getLogger('cmdstanpy').disabled = True      # Prophet 학습 로그 숨김
logging.getLogger('prophet').disabled = True
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from prophet import Prophet
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

plt.rcParams['font.family'] = 'AppleGothic'; plt.rcParams['axes.unicode_minus'] = False

train = pd.read_csv('train.csv', encoding='cp949')           # ⚠️ Train만 사용
TARGETS = ['Target_1', 'Target_2']
FEATS = [c for c in train.columns if c not in TARGETS + ['Time_step', 'Weekday']]
FEATS = [c for c in FEATS if train[c].nunique() > 1]          # 상수 컬럼(KE_QR체크인) 제외
GROUP = {'FI': '금융', 'KE': '포털키워드', 'SH1': '쇼핑L1', 'SH2': '쇼핑L2', 'WE': '날씨'}
grp = pd.Series({c: GROUP[c.split('_')[0]] for c in FEATS})
X = train[FEATS]

# 가상 날짜: Weekday 1=일 … 7=토 (§1 근거). Time_step 1의 Weekday=3(화) → 2019-01-01(화)에 맞춘다
WD_NAME = {1: '일', 2: '월', 3: '화', 4: '수', 5: '목', 6: '금', 7: '토'}
train['ds'] = pd.Timestamp('2019-01-01') + pd.to_timedelta(train.Time_step - 1, 'D')
assert ((train.ds.dt.dayofweek + 1) % 7 + 1 == train.Weekday).all(), '가상 날짜의 요일이 Weekday와 불일치'

def fit_prophet(y, ds, yearly=True):
    """Prophet 학습: 요일(weekly)·연(yearly) 계절성 + 추세. 일중(daily) 패턴은 일 단위 데이터라 끔"""
    m = Prophet(weekly_seasonality=True, yearly_seasonality=yearly, daily_seasonality=False)
    return m.fit(pd.DataFrame({'ds': ds, 'y': y}))

def auc(x, y, min_minor=15):
    """단일 변수 AUC. 결측 제외, 소수 클래스가 min_minor개 미만이면 계산하지 않음 (표본이 작으면 AUC가 크게 흔들림)"""
    m = x.notna() & y.notna()
    if x[m].nunique() < 2 or min(y[m].sum(), (1 - y[m]).sum()) < min_minor: return np.nan
    return roc_auc_score(y[m], x[m])

print(train.shape, '| 변수군:', grp.value_counts().to_dict())
print('가상 날짜 범위:', train.ds.min().date(), '~', train.ds.max().date())

## 1. 요일 번호의 정체 — `Weekday` 1~7은 무슨 요일인가?

설명서에는 "요일 Index"라고만 되어 있다. 요일 흐름을 해석하려면 먼저 **어느 숫자가 주말인지** 알아야 한다.

**판단 방법**: 주식·환율 시장은 주말에 쉰다 → 주말에는 금융 값이 **전날 값 그대로 복사**되어 전일 대비 변화가 0이 된다. 요일별로 '변화 = 0' 비율을 세어 보면 주말이 드러난다.

In [ ]:
fi = X.filter(like='FI_')
no_move = (fi.diff() == 0)                                   # 전일 대비 변화 없음 (값 복사)
wd_tbl = pd.DataFrame({
    '요일': pd.Series(WD_NAME),
    '금융 변화=0 비율': no_move.groupby(train.Weekday).mean().mean(axis=1),
    '미국지수 변화=0': no_move[['FI_Sp500', 'FI_Dow', 'FI_Nasdaq']].groupby(train.Weekday).mean().mean(axis=1),
    'T1 상승비율': train.groupby('Weekday').Target_1.mean(),
    'T2 상승비율': train.groupby('Weekday').Target_2.mean(),
    '식품 클릭(평균=1)': train.groupby('Weekday')['SH1_쇼핑1_식품'].mean() / train['SH1_쇼핑1_식품'].mean(),
})
wd_tbl['3일 뒤 요일'] = [WD_NAME[(d + 2) % 7 + 1] for d in wd_tbl.index]   # D+3의 요일
wd_tbl.round(2)

#### 👀 표 읽는 법
- **금융 변화=0 비율이 요일 7·1에서 0.89 이상**(미국 지수는 1.00)이고 나머지 요일은 0.1 안팎이다 → **7 = 토요일, 1 = 일요일**. 엑셀 `WEEKDAY()` 기본값(1=일 … 7=토)과 같은 체계다.
- **T2를 요일 이름으로 다시 읽으면** (`3일 뒤 요일` 열과 같이 본다):
  - 상승 ≈ 0.95: **일→수, 금→월, 토→화** = 주말에서 평일로 넘어가는 구간
  - 하락 (상승 ≈ 0.03): **화→금, 수→토** = 평일에서 주말로 넘어가는 구간
  - 애매 (≈ 0.18): **월→목, 목→일** = 둘 다 평일이거나 변화가 작은 구간
  - → 사람들은 **평일에 쇼핑을 더 많이 클릭**한다(식품 클릭 월 1.21 vs 토 0.69). 건강식품 클릭 지수도 같은 패턴이라 "3일 뒤가 더 '평일다운' 날인가"가 곧 T2다.
- **T1은 요일과 무관**(모든 요일 ≈ 0.5)하다. 다만 주말 행의 금융 값은 금요일 복사본이라, `chg1` 같은 '전일 대비' 피처가 주말에는 항상 0이 된다 → 주의가 필요하다.

## 2. 변수군별 Prophet 분해 — 각 군은 추세·요일·계절 중 무엇으로 움직이나

**방법**
1. **군 지수 만들기**: 군에 속한 변수를 각각 표준화(평균 0, 표준편차 1)한 뒤 평균한다 → 군 전체의 공통 움직임 하나. 표준화하는 이유는 변수마다 크기가 달라 큰 변수가 평균을 지배하지 않게 하려는 것이다.
2. **Prophet으로 분해**: 군 지수 = 추세 + 요일 + 연 계절성 + 잔차
3. **분산 비율**: 각 성분의 분산 ÷ 원래 분산 → "이 군의 움직임 중 몇 %가 요일 때문인가"
   - 성분끼리 완전히 독립은 아니라서 합이 정확히 1이 되지는 않는다. **상대 비교용**으로 본다.

In [ ]:
Z = (X - X.mean()) / X.std()                                  # 변수별 표준화
decomp, comps = [], {}
for name in GROUP.values():
    y = Z[grp[grp == name].index].mean(axis=1)                 # ① 군 지수
    fc = fit_prophet(y, train.ds).predict(train[['ds']])       # ② 분해 (Train 전체로 학습 = 분석용)
    comps[name] = fc.assign(y=y.values)
    v = y.var()                                                 # ③ 분산 비율
    decomp.append({'군': name, '변수 수': (grp == name).sum(), '추세': fc.trend.var() / v,
                   '요일': fc.weekly.var() / v, '연계절': fc.yearly.var() / v, '잔차(설명 안 됨)': (y - fc.yhat).var() / v})
decomp = pd.DataFrame(decomp).set_index('군')

fig, ax = plt.subplots(len(GROUP), 3, figsize=(18, 14))
for i, (name, fc) in enumerate(comps.items()):
    ax[i, 0].plot(fc.ds, fc.y, c='lightgray', lw=.6, label='군 지수')
    ax[i, 0].plot(fc.ds, fc.trend, c='k', lw=2, label='추세')
    ax[i, 0].set_title(f'{name} — 추세 ({decomp.loc[name, "추세"]:.0%})'); ax[i, 0].legend(fontsize=8)
    w = fc.groupby(train.Weekday).weekly.mean()                 # 요일 성분을 요일별 평균으로
    ax[i, 1].bar([WD_NAME[d] for d in w.index], w.values, color=['tomato' if d in (1, 7) else 'steelblue' for d in w.index])
    ax[i, 1].axhline(0, c='k', lw=.8); ax[i, 1].set_title(f'{name} — 요일 성분 ({decomp.loc[name, "요일"]:.0%}, 빨강=주말)')
    ax[i, 2].plot(fc.ds, fc.yearly, c='seagreen'); ax[i, 2].axhline(0, c='k', lw=.8)
    ax[i, 2].set_title(f'{name} — 연 계절성 ({decomp.loc[name, "연계절"]:.0%})')
plt.tight_layout(); plt.show()
decomp.round(2)

#### 👀 그림·표 읽는 법
- **왼쪽**: 회색 = 실제 군 지수, 검정 = Prophet이 뽑은 추세. 검정선이 크게 움직일수록 '추세형' 군이다.
- **가운데**: 요일 성분. 막대가 0에서 멀수록 그 요일에 평소보다 높거나(+) 낮다(−). 빨강 = 토·일.
- **오른쪽**: 1년 주기 성분 (가상 날짜라 '몇 월'인지는 의미 없고 **주기가 있는지**만 본다).
- 표의 % = 각 성분이 군의 움직임을 설명하는 비율.

#### 변수군별 해석
| 군 | 주된 성분 | 의미 → 피처 설계 |
| --- | --- | --- |
| **금융** | 추세 ≈ 66%, 요일 ≈ 0% | 수개월 단위로 오르내림. 요일 패턴은 없고 **주말엔 값이 멈춤**(복사) → 원값 말고 **변화율**, 주말은 '마지막 거래일 변화' 필요 |
| **포털키워드** | **요일 ≈ 54%** | 검색량은 평일/주말 차이가 가장 크다 → 키워드 원값·변화율의 신호 상당수가 **요일의 다른 표현**일 수 있음 (§3에서 검증) |
| **쇼핑 L1·L2** | 추세 ≈ 40%, 요일 18~29% | 이용량이 점점 늘어나는 추세 + **일~화 높고 금·토 낮은** 요일 패턴 → 원값은 추세 때문에 시기별로 달라지고, 요일 차이도 커서 **같은 요일끼리 비교(`sameWD`, `d3_ratio`)** 가 필요 |
| **날씨** | 잔차 ≈ 88% | 군 평균으로는 거의 설명 안 됨 (기온·미세먼지·풍향이 서로 상쇄). 개별 변수로 봐야 함 (§2-2) |

### 2-2. 대표 변수 하나씩 — 군 평균에서 상쇄된 패턴 확인
군 지수는 성격이 다른 변수가 섞이면 패턴이 상쇄된다(특히 날씨). 군별 대표 변수 1개씩을 따로 분해한다.

In [ ]:
REP = {'금융': 'FI_Sp500', '포털키워드': 'KE_보험', '쇼핑L1': 'SH1_쇼핑1_식품', '날씨': 'WE_평균기온'}
fig, ax = plt.subplots(1, 4, figsize=(20, 4)); rows = []
for a, (g, c) in zip(ax, REP.items()):
    fc = fit_prophet(X[c], train.ds).predict(train[['ds']]); v = X[c].var()
    rows.append({'군': g, '변수': c, '추세': fc.trend.var() / v, '요일': fc.weekly.var() / v, '연계절': fc.yearly.var() / v})
    w = fc.groupby(train.Weekday).weekly.mean()
    a.bar([WD_NAME[d] for d in w.index], w.values, color=['tomato' if d in (1, 7) else 'steelblue' for d in w.index])
    a.axhline(0, c='k', lw=.8); a.set_title(f'{c}\n요일 성분 (요일 {rows[-1]["요일"]:.0%} · 연 {rows[-1]["연계절"]:.0%})', fontsize=10)
plt.tight_layout(); plt.show()
pd.DataFrame(rows).set_index('군').round(2)

#### 👀 읽는 법
- `FI_Sp500`: 요일 막대가 거의 0 → 주가에는 요일 패턴이 없다. 주말(빨강)에 값이 멈춰 있을 뿐이다.
- `KE_보험`: 평일 높고 주말 낮은 전형적 '업무 시간' 검색 패턴.
- `SH1_쇼핑1_식품`: 월~수 높고 토요일이 가장 낮다 → **T2(건강식품 클릭)의 대리변수**이고 §1 표와 같은 모양이다.
- `WE_평균기온`: 요일 성분 ≈ 0, **연 계절성이 대부분** → 날씨는 요일이 아니라 계절로 움직인다. Target은 계절 주기가 없으므로 날씨 신호가 약한 이유와 연결된다.

## 3. 변수군 × Target 관계 — 어떻게 확인했나 (01 EDA의 방법 + 요일 통제)

변수 280여 개를 하나씩 눈으로 볼 수는 없으므로, **모든 변수를 같은 자로 재고 군 단위로 모아 본다.**

| 단계 | 하는 일 | 왜 | 01 EDA 위치 |
| --- | --- | --- | --- |
| ① 같은 자 | 변수 하나로 Target을 얼마나 가르는지 **AUC**로 잰다 (0.5 = 무작위, 멀수록 강함) | 평가지표와 같은 자. 방향(+/−) 무관하게 `|AUC−0.5|`로 세기 비교 | §3 |
| ② 형태 바꾸기 | 원값 / 1·3일 변화율 / 같은 요일 대비(`sameWD`)로 바꿔 각각 잰다 | 같은 변수도 '수준'보다 '변화'에 신호가 있을 수 있음 | §3 |
| ③ 안정성 | Train **앞 절반·뒤 절반**에서 방향이 같은지 본다 | 한 시기에만 우연히 맞은 신호 걸러내기 | §3-2 |
| ④ **요일 통제** (이번에 추가) | **같은 요일끼리만** AUC를 재서 평균한다 | 신호가 '요일을 다시 말한 것'인지, **요일과 별개의 정보**인지 구분 | §4를 일반화 |

**④가 핵심이다.** 예를 들어 쇼핑 클릭의 3일 변화율이 T2와 AUC 0.07로 강해도, 그건 "평일→주말이면 클릭이 줄고 T2도 하락"이라는 **요일 효과**일 수 있다. 같은 요일끼리만 비교하면 요일 효과가 사라지고, 그래도 남는 신호만 **진짜 추가 정보**다.
- 같은 요일 안에서 소수 클래스가 15개 미만이면 AUC가 너무 흔들려서 제외한다 → T2는 상승/하락이 섞인 **월·목(2·5)** 위주로 계산되고, T1은 7개 요일 모두 쓴다.

In [ ]:
def same_wd(D):                    # 지난 4주 같은 요일 평균 대비 (과거 값만)
    return D / (sum(D.shift(7 * k) for k in range(1, 5)) / 4)

TF = {'원값': X, 'chg1': X.pct_change(1, fill_method=None), 'chg3': X.pct_change(3, fill_method=None), 'sameWD': same_wd(X)}
TF = {k: D.replace([np.inf, -np.inf], np.nan) for k, D in TF.items()}
half = len(train) // 2
wd = train.Weekday

rows = []
t0 = time.time()
for tf, D in TF.items():
    for t in TARGETS:
        y = train[t]
        for c in FEATS:
            x = D[c]
            a_all, a1, a2 = auc(x, y), auc(x[:half], y[:half]), auc(x[half:], y[half:])       # ①②③
            per_wd = {d: (auc(x[wd == d], y[wd == d]), (wd == d).sum()) for d in range(1, 8)}  # ④ 요일별 AUC
            ok = [(a, n) for a, n in per_wd.values() if not np.isnan(a)]
            a_wd = np.average([a for a, _ in ok], weights=[n for _, n in ok]) if ok else np.nan
            rows.append((c, grp[c], tf, t, a_all, a1, a2, a_wd))
res = pd.DataFrame(rows, columns=['feat', 'group', 'tf', 'target', 'auc', 'auc_h1', 'auc_h2', 'auc_wd'])
same_dir = np.sign(res.auc_h1 - .5) == np.sign(res.auc_h2 - .5)
res['신호(전체)'] = (res.auc - .5).abs()
res['안정'] = same_dir & ((res.auc_h1 - .5).abs() > .03) & ((res.auc_h2 - .5).abs() > .03)
res['신호(요일통제)'] = (res.auc_wd - .5).abs()
print(f'{len(res):,}개 조합 계산 ({time.time() - t0:.0f}초)')

summary = res.groupby(['target', 'group']).agg(
    최대신호_전체=('신호(전체)', 'max'), 최대신호_요일통제=('신호(요일통제)', 'max'),
    안정신호_비율=('안정', 'mean'), 요일통제후_신호005_비율=('신호(요일통제)', lambda s: (s > .05).mean()))

fig, ax = plt.subplots(1, 2, figsize=(16, 4.5))
for a, t in zip(ax, TARGETS):
    s = summary.loc[t]
    xx = np.arange(len(s)); a.bar(xx - .2, s.최대신호_전체, .4, label='전체 (요일 효과 포함)', color='#9e9e9e')
    a.bar(xx + .2, s.최대신호_요일통제, .4, label='같은 요일끼리 (요일 효과 제거)', color='#1f77b4')
    a.set_xticks(xx, s.index); a.set_ylabel('최대 |AUC - 0.5|'); a.legend(fontsize=9)
    a.set_title(f'{t}: 변수군별 가장 강한 신호 — 요일 효과를 빼면 얼마나 남나')
plt.tight_layout(); plt.show()
summary.round(3)

#### 👀 그림 읽는 법
막대 높이 = 그 군에서 **가장 강한 단일 변수 신호**. 회색(요일 포함) → 파랑(같은 요일끼리)으로 **많이 줄어들면 그 신호는 대부분 요일 효과**, 비슷하면 **요일과 별개인 진짜 정보**다.

#### 변수군 × Target 관계 정리
| 군 | Target_1 (KOSPI) | Target_2 (건강식품 클릭) |
| --- | --- | --- |
| **금융** | 가장 강하고 **요일 통제 후에도 유지**(오히려 ↑): 미국 지수 `chg1` → 직전 미국장 등락이 KOSPI를 이끈다는 **실제 정보** | 약함 — 주가와 쇼핑 클릭은 무관 |
| **포털키워드** | 약함 (대부분 잡음) | 전체 0.43 → 요일 통제 0.25: **절반 가까이는 요일 효과**. 그래도 0.25(AUC 0.75 수준)가 남는다 → 보험 키워드 등은 월·목 안에서도 통하는 보강 재료 |
| **쇼핑 L1·L2** | 약함. `sameWD`(평소 대비) 일부만 | 전체 0.40~0.43 → 요일 통제 0.25 수준. **요일 + α**: 요일이 큰 몫이지만 α(월·목 안에서의 차이)도 상당 → `d3_ratio`가 요일 + 최근 클릭 수준을 함께 담는 이유 |
| **날씨** | 약함 | 약함 (요일 통제 후 일부 튀는 값은 월·목 264일 안의 우연일 가능성 — 안정성 확인 필요) |

> 표의 `안정신호_비율` = 앞/뒤 절반 모두 같은 방향으로 |AUC−0.5| > 0.03인 변수 비율. 낮을수록 잡음이 많다는 뜻이다.
> `요일통제후_신호005_비율`은 T2에서 월·목 약 260일만으로 계산되어 우연 오차(±0.05 수준)도 섞여 있다. 비율보다는 **최대 신호의 감소 폭**을 위주로 본다.

In [ ]:
# 군별로 '요일 통제 후에도 남는' 상위 변수 — 어떤 변수가 진짜 추가 정보인가
top = (res[res.안정].sort_values('신호(요일통제)', ascending=False)
          .groupby(['target', 'group']).head(2)
          .sort_values(['target', '신호(요일통제)'], ascending=[True, False]))
top[['target', 'group', 'feat', 'tf', 'auc', 'auc_wd']].rename(columns={'auc': 'AUC(전체)', 'auc_wd': 'AUC(같은 요일끼리)'}).round(3)

#### 👀 표 읽는 법
- 앞/뒤 절반에서 방향이 같은(안정) 변수 중, **요일 효과를 뺀 뒤에도 신호가 강한** 상위 2개씩이다.
- `AUC(전체)`와 `AUC(같은 요일끼리)`가 **같은 쪽(둘 다 0.5 위 또는 아래)이면** 요일과 별개로도 통하는 신호다. 전체는 0.1인데 요일끼리는 0.4처럼 **0.5 쪽으로 크게 줄면** 대부분 요일 효과였다는 뜻이다.

## 4. Prophet을 예측에 써 보기 — 시계열 CV로 공정하게 비교

**설계** (train.ipynb와 같은 `TimeSeriesSplit(5, gap=3)`)
- 각 fold: Prophet을 **fold Train 날짜까지만** 학습 → fold Valid 기간을 예측 → AUC.
- gap=3: Train 마지막 날의 Target(3일 뒤 값)이 Valid 기간과 겹치지 않게 3일을 비운다.

| 실험 | 내용 | 의도 |
| --- | --- | --- |
| **P1** | Target(0/1)을 그대로 Prophet에 넣어 예측 | 가장 단순한 시계열 기준선. T2는 요일 성분이, T1은 추세·연 계절이 점수를 만든다 |
| P1-연계절X | P1에서 연 계절성 끔 (T1) | 2.5년 데이터로 '1년 주기'를 배우는 게 믿을 만한가 확인 |
| 요일만 | fold Train의 요일별 상승비율 | T2 비교 기준 |
| **P2-ratio** | 대리변수(식품+생활건강 클릭) Prophet 예측 → **3일 뒤 예측값 ÷ 오늘 실제값** | T2 정의("3일 뒤 클릭 > 오늘?")를 그대로 흉내 |
| P2-weekly | 같은 Prophet의 **요일 성분 차이**(3일 뒤 요일 − 오늘 요일) | 추세 오차 없이 요일 패턴만 |
| d3_ratio | 기존 수작업 피처 (지난 4주 같은 요일 평균 ÷ 오늘) | 현재 train.ipynb의 핵심 피처 |
| LR(요일+d3) / +P2 | 로지스틱 회귀에 P2를 **추가**했을 때 오르나 | Prophet 피처의 **추가 가치** |

In [ ]:
TSCV = TimeSeriesSplit(n_splits=5, gap=3)
food = train[['SH1_쇼핑1_식품', 'SH1_쇼핑1_생활건강']].mean(axis=1)            # T2 대리변수 (건강식품의 상위 분류)
d3 = (sum(food.shift(4 + 7 * k) for k in range(4)) / 4) / food.replace(0, np.nan)   # 기존 d3_ratio
wd_oh = pd.get_dummies(train.Weekday, prefix='wd').astype(int)
y1, y2 = train.Target_1, train.Target_2
D3 = pd.Timedelta(days=3)

def lr():
    return make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), LogisticRegression(C=0.1, max_iter=2000))

scores = {}
def add(k, v): scores.setdefault(k, []).append(v)

t0 = time.time()
for tr, va in TSCV.split(train):
    ds_tr, ds_va = train.ds.iloc[tr], train.ds.iloc[va]
    # P1: Target 직접
    add('T1 | P1 Prophet(Target 직접)', roc_auc_score(y1.iloc[va], fit_prophet(y1.iloc[tr], ds_tr).predict(pd.DataFrame({'ds': ds_va})).yhat))
    add('T1 | P1 연계절 끔', roc_auc_score(y1.iloc[va], fit_prophet(y1.iloc[tr], ds_tr, yearly=False).predict(pd.DataFrame({'ds': ds_va})).yhat))
    add('T2 | P1 Prophet(Target 직접)', roc_auc_score(y2.iloc[va], fit_prophet(y2.iloc[tr], ds_tr).predict(pd.DataFrame({'ds': ds_va})).yhat))
    add('T2 | 요일만', roc_auc_score(y2.iloc[va], train.Weekday.iloc[va].map(y2.iloc[tr].groupby(train.Weekday.iloc[tr]).mean())))
    # P2: 대리변수 예측 (fold Train까지만 학습 → 전 기간 + 3일 예측)
    fc = fit_prophet(food.iloc[tr], ds_tr).predict(pd.DataFrame({'ds': pd.date_range(train.ds.min(), train.ds.max() + D3)})).set_index('ds')
    p2_ratio = pd.Series(fc.yhat.reindex(train.ds + D3).values / food.replace(0, np.nan).values)    # 3일 뒤 예측 ÷ 오늘 실제(이미 아는 값)
    p2_week = pd.Series(fc.weekly.reindex(train.ds + D3).values - fc.weekly.reindex(train.ds).values)
    add('T2 | P2 Prophet 대리예측 ratio', roc_auc_score(y2.iloc[va], p2_ratio.iloc[va].fillna(1)))
    add('T2 | P2 Prophet 요일성분 차이', roc_auc_score(y2.iloc[va], p2_week.iloc[va]))
    add('T2 | d3_ratio (기존 피처)', roc_auc_score(y2.iloc[va], d3.iloc[va].fillna(1)))
    for name, parts in {'T2 | LR 요일+d3': [wd_oh, d3.rename('d3')],
                        'T2 | LR 요일+d3 + P2': [wd_oh, d3.rename('d3'), p2_ratio.rename('p2')]}.items():
        Xd = pd.concat(parts, axis=1)
        add(name, roc_auc_score(y2.iloc[va], lr().fit(Xd.iloc[tr], y2.iloc[tr]).predict_proba(Xd.iloc[va])[:, 1]))
print(f'학습 시간 {time.time() - t0:.0f}초')

cv = pd.DataFrame({k: {'평균': np.mean(v), '표준편차': np.std(v), '평균−std': np.mean(v) - np.std(v),
                       'folds': np.round(v, 3)} for k, v in scores.items()}).T
REF = {'T1': ('train.ipynb E6 (RF)', 0.5987, 0.0201), 'T2': ('train.ipynb E5 (앙상블)', 0.9665, 0.0105)}

fig, ax = plt.subplots(1, 2, figsize=(18, 5), gridspec_kw={'width_ratios': [2, 5]})
for a, t in zip(ax, ['T1', 'T2']):
    d = cv[cv.index.str.startswith(t)]
    names = [n.split('| ')[1] for n in d.index] + [REF[t][0]]
    m = list(d['평균'].astype(float)) + [REF[t][1]]; s = list(d['표준편차'].astype(float)) + [REF[t][2]]
    colors = ['#ff7f0e' if 'Prophet' in n else '#9e9e9e' for n in names[:-1]] + ['#1f77b4']
    a.bar(range(len(m)), m, yerr=s, capsize=4, color=colors)
    a.set_xticks(range(len(m)), names, rotation=25, ha='right', fontsize=9)
    a.set_ylim(.45 if t == 'T1' else .85, None); a.axhline(.5, c='k', lw=.6, ls=':')
    a.set_title(f'Target_{t[-1]}: CV Valid AUC (주황 = Prophet, 파랑 = 현재 최종 모델)')
plt.tight_layout(); plt.show()
cv

#### 👀 그림 읽는 법
막대 = 5개 fold 평균 AUC, 검은 선 = fold 간 표준편차(길수록 시기마다 들쭉날쭉). **주황 = Prophet 기반**, 회색 = 비교 기준, **파랑 = 현재 train.ipynb 최종 모델**.

#### 결과 해석
| 질문 | 답 |
| --- | --- |
| Prophet이 T2의 요일 구조를 잡나? | ✅ P1(Target 직접)이 '요일만'과 비슷한 수준 → Prophet의 **요일 성분 = 요일별 상승비율**을 자동으로 배운 것 |
| Prophet 대리예측(P2)이 기존 `d3_ratio`보다 낫나? | ✗ P2 ≈ 0.947 < `d3_ratio` ≈ 0.954. Prophet은 몇 달 앞까지 **한 번 학습한 추세를 그대로 연장**하므로, '지난 4주 실제값'을 쓰는 `d3_ratio`보다 최근 변화를 덜 반영한다 |
| P2를 추가하면 오르나? | ✗ LR 요일+d3 → +P2: +0.001 수준 (잡음 범위). `d3_ratio`와 **같은 정보**(요일 패턴)를 다른 방식으로 표현한 것이기 때문 |
| T1에 쓸 만한가? | ✗ P1 ≈ 0.56이지만 표준편차가 ±0.05로 크고 fold별로 0.49~0.63을 오간다. **연 계절성을 끄면 0.49(무작위)로 떨어진다** → 점수가 전부 '1년 주기'에서 나왔다. Train은 2.5년(주기 2.5번 반복)뿐이라 이 주기는 **우연일 가능성**이 크다 → 채택 근거 부족 |

**결론**: Prophet은 이 과제에서 **분석·설명 도구(§2 분해)로는 유용**하지만, **예측 성능은 현재 모델(파랑)을 넘지 못한다**. 이유는 두 가지다.
1. T2의 핵심인 요일 패턴은 이미 `wd_onehot` + `d3_ratio`가 더 직접적으로 잡고 있다.
2. T1(KOSPI)에는 Prophet이 잡을 추세·계절 구조가 거의 없다.

보고서에는 "Prophet을 시도해 **요일 성분이 T2의 핵심임을 독립적으로 재확인**했고, 예측 기여는 없어 최종 모델에서 제외"로 쓰면 된다.

## 5. 요일별로 바뀌는 흐름을 어떻게 고려했나 — 총정리

| 단계 | 무엇을 했나 | 무엇을 알았나 | 어디에 반영 |
| --- | --- | --- | --- |
| ① 요일별 상승비율 | Target을 요일로 묶어 평균 | T2는 일·금·토 ≈ 0.95, 화·수 ≈ 0.03 / T1은 요일 무관 | `wd_onehot` (두 Target) — 01 EDA §1 |
| ② Target 자기상관 | 1~14일 전 Target과 상관 | T2가 정확히 7일 주기 | 요일 구조 확신 — 01 EDA §1 |
| ③ 같은 요일끼리 비교 변환 | `sameWD` = 지난 4주 **같은 요일** 평균 대비 | 요일 효과를 지우면 T2 신호 대부분이 사라짐 → 신호의 정체 = 요일 | T1 `samewd_grp` — 01 EDA §3-1 |
| ④ **D+3 요일** 파생변수 | `d3_ratio` = 지난 4주 '3일 뒤와 같은 요일' 평균 ÷ 오늘 | 요일만(0.93)보다 높은 0.96 — 요일 + 최근 수준까지 반영 | T2 `d3ratio` 블록 — 01 EDA §8 |
| ⑤ 애매한 요일만 따로 | 월·목(2·5)만 모아 신호 탐색 | 요일로 못 가르는 날은 보험 키워드 등 **평균회귀**(`z20`)가 보강 | T2 `z20_kw` — 01 EDA §4 |
| ⑥ lag 히트맵 | 1~10일 전 값의 신호 | T2 신호가 7일 주기로 부호 반전 = 변수 정보가 아니라 요일 위치 | 상위 `chg3` 대량 투입 금지 — 01 EDA §5 |
| ⑦ 주말 정체 확인 | 요일별 금융 '변화=0' 비율 | **7=토, 1=일**. 주말 금융값은 금요일 복사 → 주말 `chg1` = 0 | 주말엔 '마지막 거래일 변화' 사용 제안 (E7 후보) — 본 노트북 §1 |
| ⑧ Prophet 요일 성분 | 군별 요일 성분 분산 비율 | 포털 54%, 쇼핑 18~29%, 금융 0%, 날씨 1% | 군별 설명 근거 — 본 노트북 §2 |
| ⑨ 요일 통제 AUC | 같은 요일 안에서만 AUC | 금융(T1)은 요일과 별개 정보, 쇼핑·키워드(T2)는 대부분 요일 효과 | 피처 선택 근거 — 본 노트북 §3 |
| ⑩ 검증 | 모든 요일 피처를 **과거 값만**으로 만들고 시계열 CV로 평가 | 요일 피처 덕에 T2 0.945 → 0.967 | train.ipynb §4 |

**핵심 한 줄**: 요일은 **T2의 답 그 자체**(쇼핑 클릭의 평일/주말 주기)이고, **T1에는 잡음이거나 함정**(주말 금융값 복사)이다. 그래서 T2는 요일을 적극적으로 피처로 넣었고, T1은 요일 효과를 지운 뒤에도 남는 신호(미국장 등락)만 골랐다.